# 03-1 baseline - no summary training

이 노트북에서 하는 일: **파인튜닝하지 않은 KoBART**에 뉴스 원문 `document`만 입력해 요약을 생성한다.

이 실험은 수업의 사전학습 BART와 `model.generate()` 사용법을 적용한 zero-shot 기준선입니다.

- 학습하지 않는다.
- optimizer와 `loss.backward()`를 사용하지 않는다.
- 모델 입력에는 `document`만 들어간다.
- 데이터의 `summary`는 생성이 끝난 뒤 ROUGE 평가와 사례 비교에만 사용한다.
- 원본 test 2,740건을 그대로 사용한다.

따라서 이 실험의 점수는 이후 KoBART 파인튜닝 결과와 비교할 **학습 전 성능**이다.


## 1. 환경 및 생성 설정

In [1]:
import os
import sys
from pathlib import Path

PROJECT = Path(r"C:\Users\KDS-30\Desktop\Projects\06_Team_Project4")
assert PROJECT.exists(), "프로젝트 경로를 확인하세요."
assert Path(sys.prefix).resolve() == (PROJECT / ".venv").resolve(), (
    "커널을 Python 3.11 (Team4 News GPU)로 바꾸세요."
)

os.environ["HF_HOME"] = str(PROJECT / ".cache" / "huggingface")
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

RESULTS = PROJECT / "results" / "03-1_baseline_no_summary"
RESULTS.mkdir(parents=True, exist_ok=True)

SEED = 42
DATASET_ID = "daekeun-ml/naver-news-summarization-ko"
MODEL_ID = "gogamza/kobart-base-v2"
MAX_SOURCE_LENGTH = 512
MAX_NEW_TOKENS = 128
BATCH_SIZE = 4
RUN_GENERATION = True
MAX_EVAL_SAMPLES = None  # 빠른 점검은 20처럼 지정, 전체 평가는 None

print("Python:", sys.version.split()[0])
print("결과 폴더:", RESULTS)

Python: 3.11.9
결과 폴더: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\results\00_baseline_no_summary


In [2]:
import json
import random
import time

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import HfApi
from torch.utils.data import DataLoader, SequentialSampler, TensorDataset
from transformers import AutoTokenizer, BartForConditionalGeneration

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "CUDA를 사용할 수 없습니다."
print("device:", device)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM(GB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM(GB): 7.96


## 2. 원본 test 데이터 로드

In [3]:
api = HfApi()
dataset_revision = api.dataset_info(DATASET_ID).sha
model_revision = api.model_info(MODEL_ID).sha
raw_dataset = load_dataset(DATASET_ID, revision=dataset_revision)

test_df = raw_dataset["test"].to_pandas().reset_index(drop=True)
test_df.insert(0, "source_index", np.arange(len(test_df), dtype=np.int64))
test_df.insert(0, "row_id", "test:" + test_df["source_index"].astype(str))

assert len(test_df) == 2740
assert test_df["document"].notna().all()
assert test_df["document"].str.strip().ne("").all()

if MAX_EVAL_SAMPLES is not None:
    evaluation_df = test_df.iloc[:MAX_EVAL_SAMPLES].copy()
else:
    evaluation_df = test_df.copy()

print("dataset revision:", dataset_revision)
print("model revision:", model_revision)
print("원본 test:", len(test_df))
print("이번 실행 대상:", len(evaluation_df))
display(evaluation_df[["title", "document"]].head(3))

dataset revision: 916dfe6aed651b2ecd9d04fa0a704f8953e90762
model revision: f5899cc556216353e23ee36b66fad52b902066c5
원본 test: 2740
이번 실행 대상: 2740


,title,document
0,아이트로닉스 차량용 복합기능형 졸음 방지 단말기 특허 출원,아이엘사이언스의 자회사 아이트로닉스는 차량용 복합기능형 졸음 방지 단말기 특허를 출...
1,뉴스페이스로 가는 한국… 우주기업 성장할 기술·터전 만든다,한국형 우주발사체 누리호 가 지난 6월 21일 전남 고흥 나로우주센터 발사대에서 날...
2,SP500 올 상반기 21%↓…50년래 최악 월가월부,닥터둠 루비니 주가 50% 폭락할 것 뉴욕 증시 3대지수 모두 하락 대형주 중심의 ...


## 3. document만 토큰화

아래 `TensorDataset`에는 `input_ids`와 `attention_mask`만 존재합니다. 정답 `summary` tensor는 만들지 않습니다.


In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=model_revision,
    clean_up_tokenization_spaces=False,
)

def make_document_dataset(data, tokenizer):
    source = tokenizer(
        text=data["document"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        return_token_type_ids=False,
        return_tensors="pt",
    )
    return TensorDataset(source["input_ids"], source["attention_mask"])

def get_dataloader(dataset, batch_size):
    return DataLoader(
        dataset,
        sampler=SequentialSampler(dataset),
        batch_size=batch_size,
        num_workers=0,
        pin_memory=True,
    )

document_dataset = make_document_dataset(evaluation_df, tokenizer)
document_dataloader = get_dataloader(document_dataset, BATCH_SIZE)

sample_batch = next(iter(document_dataloader))
assert len(sample_batch) == 2, "입력에는 input_ids와 attention_mask만 있어야 합니다."
print("input_ids:", sample_batch[0].shape)
print("attention_mask:", sample_batch[1].shape)
print("summary는 모델 입력에 포함되지 않았습니다.")

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


input_ids: torch.Size([4, 512])
attention_mask: torch.Size([4, 512])
summary는 모델 입력에 포함되지 않았습니다.


## 4. 파인튜닝하지 않은 KoBART 로드

In [5]:
model = BartForConditionalGeneration.from_pretrained(
    MODEL_ID,
    revision=model_revision,
    use_safetensors=True,
).to(device)
model.eval()

print("model:", MODEL_ID)
print("parameters:", f"{sum(parameter.numel() for parameter in model.parameters()):,}")
print("학습 또는 checkpoint 로드는 수행하지 않았습니다.")

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


model: gogamza/kobart-base-v2
parameters: 123,859,968
학습 또는 checkpoint 로드는 수행하지 않았습니다.


## 5. 원문만으로 요약 생성

첫 zero-shot 기준선은 가장 단순한 greedy decoding을 사용합니다. beam search와 반복 억제는 이후 디코딩 전략 비교에서 별도로 실험합니다.


In [6]:
def generate_summaries(model, dataloader):
    predictions = []
    started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()

    with torch.no_grad():
        for step, (input_ids, attention_mask) in enumerate(dataloader, start=1):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)

            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=1,
                do_sample=False,
            )
            predictions.extend(
                tokenizer.batch_decode(
                    generated_ids,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=False,
                )
            )

            if step % 100 == 0:
                print(f"  generation {step:,}/{len(dataloader):,}")

    elapsed_seconds = time.perf_counter() - started
    peak_memory_mib = torch.cuda.max_memory_allocated() / 2**20
    return [text.strip() for text in predictions], elapsed_seconds, peak_memory_mib

if RUN_GENERATION:
    predictions, generation_seconds, peak_memory_mib = generate_summaries(
        model, document_dataloader
    )
    assert len(predictions) == len(evaluation_df)
    print("생성 완료:", len(predictions))
    print("생성 시간(분):", round(generation_seconds / 60, 2))
    print("최대 GPU tensor 메모리(MiB):", round(peak_memory_mib, 1))
else:
    print("RUN_GENERATION=False: 생성을 건너뜁니다.")

  generation 100/685
  generation 200/685
  generation 300/685
  generation 400/685
  generation 500/685
  generation 600/685
생성 완료: 2740
생성 시간(분): 6.75
최대 GPU tensor 메모리(MiB): 607.7


## 6. 생성 후 raw summary와 비교

이 단계에서 처음으로 `summary`를 참조합니다. summary는 모델 입력이나 학습에 사용되지 않고, 생성 결과의 평가 기준으로만 사용됩니다.


In [7]:
if RUN_GENERATION:
    result_df = evaluation_df[[
        "row_id", "title", "document", "summary", "category", "press", "link"
    ]].copy()
    result_df["prediction"] = predictions

    rouge_metric = evaluate.load("rouge")
    rouge_scores = rouge_metric.compute(
        predictions=result_df["prediction"].tolist(),
        references=result_df["summary"].astype(str).tolist(),
        tokenizer=lambda text: text.split(),
        use_stemmer=False,
    )
    metrics = {
        **{name: score * 100 for name, score in rouge_scores.items()},
        "evaluated_rows": len(result_df),
        "generation_seconds": generation_seconds,
        "peak_memory_mib": peak_memory_mib,
    }

    result_df.to_csv(
        RESULTS / "zero_shot_test_predictions.csv", index=False, encoding="utf-8-sig"
    )
    (RESULTS / "zero_shot_metrics.json").write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    display(pd.Series(metrics, name="KoBART zero-shot"))
    display(result_df[["title", "summary", "prediction"]].head(10))
else:
    print("RUN_GENERATION=False: 평가를 건너뜁니다.")

rouge1                  19.759300
rouge2                  15.104644
rougeL                  17.118359
rougeLsum               17.107262
evaluated_rows        2740.000000
generation_seconds     404.705739
peak_memory_mib        607.741699
Name: KoBART zero-shot, dtype: float64

,title,summary,prediction
0,아이트로닉스 차량용 복합기능형 졸음 방지 단말기 특허 출원,"아이일, 아이트로닉스는 차량용 복합기능형 졸음 방지 단말기 특허를 출원했다고 4일 ...",단단단다.다.다.다.다.다.다. 해당 단말다. 해당 단말기는 해당 단말기는다.다. ...
1,뉴스페이스로 가는 한국… 우주기업 성장할 기술·터전 만든다,한국형 우주발사체 누리호가 지난 6월 21일 전남 고흥 나로우주센터 발사대에서 성공...,다.다.이다.이다.다.이다.다.다.다.다.다.다.다.다.다.다.다.다.다.다.다.다...
2,SP500 올 상반기 21%↓…50년래 최악 월가월부,대 대형주 중심의 미국 스탠더드앤드푸어스 S P 500 지수가 올 상반기 50여년만...,
3,풀무원 알래스칸 명태살 ‘볼카츠’ 출시,5일 5일 풀무원식품은 알래스카 청정해역의 신선한 명태살로 만든 연육을 동그랗게 빚...,하게 튀다.다.다. 신제품 ‘볼카츠’ 2종 통통새우 에그포테이토 은 깨끗한 자연환경...
4,일동제약 생산본부장에 강덕원 부사장 영입,일동제약은 생산본부장으로 강덕원 부사장을 영입했다고 지난 1일 밝혔으며 강덕원 본부...,은은은 생산본부장으로 강덕원 부사장을 영입했다고 지난 1일 밝혔다. 강덕원 본부장은...
5,강릉커피산업 발전·애로기술 해결 의기투합…출연기관 업무협약,5일 기술회사인 커피커퍼가 KIST 강릉천연물연구소와 5일 커피 산업 발전을 위한 ...,과학지원연구원 춘천센터 농업회사법인 커피커퍼가 5일 커피 산업 발전을 위한 커피커단...
6,약세장에 펀드 회전율 5년새 최저…운용기관 방어적 운용 영향,매매에서 주식을 매매하는 빈도를 뜻하는 매매 회전율이 5년 만에 140%까지 떨어진...,다.다.다.다.다.다.다.다.다.다.다.다.다.다.다.다.다.다.다.다.다.다.다....
7,코오롱 수소 밸류체인 플랫폼 구축… 생산부터 발전까지 원스톱,코오롱그룹은 6일 동대문 디자인플라자에서 열린 ‘코리아 H2 비즈니스 서밋 2022...,사 왼쪽에서에서 네번째에서 네번째 c코오롱그룹에서 네번째 c코오롱그룹 데들에서 네번...
8,에너지정책 유턴 재생에너지 대신 원전으로,정부가 정부가 오늘 국무회의에서 의결한 새 에너지 정책 방향의 주요 내용인 원자력 ...,했습니다.했습니다.입니다. 문재인 정부가 오늘 국무회의에서 의결했습니다. 예상대로 ...
9,산업 디지털전환법 오늘부터 시행…“기업 간 협업 프로젝트 지원 확대”,산업디지털전환촉진법 시행을 하루 앞둔 지난 4일 산업부 장영진 제1차관은 산업 디지...,디지털 전환 지원산산산산산산산산산산산산산산산산산산산산산산산산산산산산산산산산산산산산산...


## 7. 실험 설정 저장

In [8]:
run_config = {
    "experiment": "KoBART zero-shot; document-only input",
    "dataset_id": DATASET_ID,
    "dataset_revision": dataset_revision,
    "evaluation_split": "original test",
    "model_id": MODEL_ID,
    "model_revision": model_revision,
    "fine_tuned": False,
    "summary_used_as_model_input": False,
    "summary_used_for_training": False,
    "summary_used_only_for_evaluation": True,
    "seed": SEED,
    "max_source_length": MAX_SOURCE_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "batch_size": BATCH_SIZE,
    "generation": {"num_beams": 1, "do_sample": False},
    "evaluation_rows": len(evaluation_df),
}
(RESULTS / "run_config.json").write_text(
    json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8"
)

for path in sorted(RESULTS.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(PROJECT))

- results\00_baseline_no_summary\run_config.json
- results\00_baseline_no_summary\zero_shot_metrics.json
- results\00_baseline_no_summary\zero_shot_test_predictions.csv
